In [1]:
import os
import time
from dotenv import load_dotenv
import requests
from datetime import datetime
import uuid
import io
import zipfile
import pandas as pd
from dateutil.relativedelta import relativedelta
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from loguru import logger
from main import APIRequests

### ------------------------------------------------- ### 

def GET_DAILY_DETAIL_HISTORY_REPORT_CSV(date: str, TOKEN_KEY: str, max_attempts: int = 5, DEF_WAIT: int = 5) -> pd.DataFrame:
    """
    Функция для получения данных из Воронки продаж
    """
        
    # === Определяем параметры первого запроса ===
    HEADERS = {'Authorization': TOKEN_KEY}
    url = 'https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads'
    report_date =  str(datetime.strptime(date, "%d.%m.%Y").strftime("%Y-%m-%d"))
    new_uuid = str(uuid.uuid4()) # Храним уникальный id отчета
    reportType = 'DETAIL_HISTORY_REPORT'
    params = {
        'startDate': report_date,
        'endDate': report_date,
        'skipDeletedNm': False
    }
    body = {
        'id': new_uuid,
        'reportType': reportType,
        'params': params
    }
    # === Первый запрос для формирования Воронки продаж ===
    logger.info(f'Получение статистики из Воронки продаж за {date}')
    for attempt in range(max_attempts):
        response = requests.post(url=url, headers=HEADERS, json=body)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 429 or STATUS_CODE in [504]:
            TIME_WAIT = DEF_WAIT * (attempt + 1)
            logger.warning(f'Ошибка {STATUS_CODE}, ожидание {TIME_WAIT} сек.')
            time.sleep(TIME_WAIT)
        elif STATUS_CODE != 200:
            logger.error(f'Ошибка запроса на формирование отчета за {date} | {STATUS_CODE} | {response.text}')
            return None
        else:
            logger.info(f'Началось формирование отчета за {date}')
            time.sleep(DEF_WAIT)
            break
        
    # === Определяем параметры второго запроса ===
    url = 'https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads'
    body = {
        'filter[downloadIds]': [new_uuid]
    }
    # === Второй запрос для проверки окончания формирования Воронки продаж ===
    logger.info(f'Проверка готовности отчета за {date}')
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS, params=body)
        STATUS_CODE = response.status_code
        
        if response.json().get('data')[0].get('status') == 'SUCCESS':
            logger.success(f'Отчет за {date} готов!')    
            break
        elif STATUS_CODE == 429: 
            if attempt == max_attempts - 1:
                logger.error(f'Ошибка проверки готовности отчета! | {STATUS_CODE} | {response.text}')
                return None
            TIME_WAIT = DEF_WAIT * (attempt + 1)
            logger.warning(f'Ошибка {STATUS_CODE} | Ожидание формирования отчета {TIME_WAIT} сек.')
            time.sleep(TIME_WAIT) 
        else:
            logger.error(f'Ошибка проверки готовности отчета! | {STATUS_CODE} | {response.text}')
            return None
        
    
    # === Определяем параметры третьего запроса ===
    url = f'https://seller-analytics-api.wildberries.ru/api/v2/nm-report/downloads/file/{new_uuid}'
    # === Третий запрос для получения отчета Воронки продаж ===    
    logger.info(f'Получение отчета за {date}')     
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 200:
            logger.success(f'Отчет за {date} получен!')
            break
        elif STATUS_CODE == 429:
            TIME_WAIT = DEF_WAIT * (attempt + 1)
            logger.warning(f'Ошибка получения отчета {STATUS_CODE} | Ожидание {TIME_WAIT} сек.')
            time.sleep(TIME_WAIT)
        else:
            logger.error(f'Ошибка получения отчета {STATUS_CODE} | {response.text}')
            return None
    # === Запрос отдает ZIP файл в котором лежит csv с названием в виде (new_uuid.csv) в побитовом формате ===
    with zipfile.ZipFile(io.BytesIO(response.content), 'r') as zip_file:
        df_day_stats = pd.read_csv(io.StringIO(zip_file.read(f'{new_uuid}.csv').decode('utf-8')), index_col=None)
        if not df_day_stats.empty:
            logger.info(f'Показатели за {date} выгружены')        
            return df_day_stats
        else:
            return None
   
### ------------------------------------------------- ###  

def CALCULATE_DAILY_DETAIL_HISTORY_REPORT_CSV(df: pd.DataFrame):
    """
    Функция расчета данных по дням из Воронки продаж
    """
    # === Рассчитываем все необходимые показатели ===
    updateDate = (datetime.now() - relativedelta(hours=3)).strftime('%d.%m.%Y %H:%M')
    reportDate = datetime.strptime(df['dt'].iloc[0], '%Y-%m-%d').strftime('%d.%m.%Y')
    ordersCount = df['ordersCount'].sum() - df['cancelCount'].sum()   
    ordersSum = df['ordersSumRub'].sum() - df['cancelSumRub'].sum()  
    buyoutsCount = df['buyoutsCount'].sum()
    buyoutsSum = df['buyoutsSumRub'].sum()
    showsCount = 0 # Placeholder (Нельзя получить по API)
    openCard = df['openCardCount'].sum()
    addToCart = df['addToCartCount'].sum()
    showToClickConversion = 0 # Placeholder (Нельзя посчитать без показов)
    addToCartConversion = (df.loc[df['addToCartConversion'] > 0, 'addToCartConversion'].mean()) / 100
    cartToOrderConversion = (df.loc[df['cartToOrderConversion'] > 0, 'cartToOrderConversion'].mean()) / 100
    buyoutPercent = (df.loc[df['buyoutPercent'] > 0, 'buyoutPercent'].mean()) / 100
    addToWishlist = df['addToWishlist'].sum()
    
    # === Собираем показатели в необходимые поля ===
    daily_stats = {
        'updateDate': updateDate,
        'reportDate': reportDate,
        'ordersCount': ordersCount,
        'ordersSum': ordersSum,
        'buyoutsCount': buyoutsCount,
        'buyoutsSum': buyoutsSum,
        'showsCount': showsCount,
        'openCard': openCard,
        'addToCart': addToCart,
        'showToClickConversion': showToClickConversion,
        'addToCartConversion': addToCartConversion,
        'cartToOrderConversion': cartToOrderConversion,
        'buyoutPercent': buyoutPercent,
        'addToWishlist': addToWishlist
    }
    return daily_stats
 
### ------------------------------------------------- ### 
           
def FORMAT_DAILY_DETAIL_HISTORY_REPORT_CSV(file: str) -> bool | None:
    """
    Функция для форматирования отчета Воронки продаж
    """
    
    # Проверяем существование файла
    file_path = Path(file)
    if not file_path.exists():
        logger.error(f'Файл {file} не найден!')
        return None
    logger.info(f'Форматирование файла {file}')
    
    try:
        # === Открываем файл ===
        df = pd.read_excel(file_path)
        
        # === Считаем размер фрейма ===
        last_row = len(df) + 1
        logger.info(f'Всего записей в файле: {last_row-1}')
        
        # === Переименовываем данные для удобства ===
        header_cols = {
        'Обновлено': 'updateDate',
        'Дата': 'reportDate',
        'Заказано, шт.': 'ordersCount',
        'Заказано, руб.': 'ordersSum',
        'Выкуплено, шт.': 'buyoutsCount',
        'Выкуплено, руб.': 'buyoutsSum',
        'Показов': 'showsCount',
        'Переходов в карточки': 'openCard',
        'Добавлений в корзину': 'addToCart',
        'Средняя конверсия в клик': 'showToClickConversion',
        'Средняя конверсия в корзину': 'addToCartConversion',
        'Средняя конверсия в заказ': 'cartToOrderConversion',
        'Средний процент выкупа': 'buyoutPercent',
        'Добавлений в Отложенные': 'addToWishlist'
        }
        df = df.rename(columns=header_cols)
        
        # === Сортируем по дате ===
        df['reportDate'] = pd.to_datetime(df['reportDate'], format='%d.%m.%Y', errors='coerce')
        df = df.sort_values('reportDate')
        df['reportDate'] = df['reportDate'].dt.strftime('%d.%m.%Y')
        logger.info(f'Данные отсортированы по дате')
        
        # === Преобразование данных к числам ===
        numeric_cols = {
            'ordersCount': int,
            'ordersSum': int,
            'buyoutsCount': int,
            'buyoutsSum': int,
            'showsCount': int,
            'openCard': int,
            'addToCart': int,
            'showToClickConversion': float,
            'addToCartConversion': float,
            'cartToOrderConversion': float,
            'buyoutPercent': float,
            'addToWishlist': int,
        }
        
        for col, dtype in numeric_cols.items():
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                if dtype == int:
                    df[col] = df[col].astype(int)
        logger.info(f'Типы данных преобразованы')
        
        # === Переименовываем столбцы ===
        header_cols = {
            'updateDate': 'Обновлено',
            'reportDate': 'Дата',
            'ordersCount': 'Заказано, шт.',
            'ordersSum': 'Заказано, руб.',
            'buyoutsCount': 'Выкуплено, шт.',
            'buyoutsSum': 'Выкуплено, руб.',
            'showsCount': 'Показов',
            'openCard': 'Переходов в карточки',
            'addToCart': 'Добавлений в корзину',
            'showToClickConversion': 'Средняя конверсия в клик',
            'addToCartConversion': 'Средняя конверсия в корзину',
            'cartToOrderConversion': 'Средняя конверсия в заказ',
            'buyoutPercent': 'Средний процент выкупа',
            'addToWishlist': 'Добавлений в Отложенные'
        }
        df = df.rename(columns=header_cols)
        logger.info(f'Столбцы переименованы')

        # === Обработка файла в xlsxwriter ===
        writer = pd.ExcelWriter(file_path, engine='xlsxwriter')
        df.to_excel(writer, index=False, sheet_name='Статистика')
        
        workbook = writer.book
        worksheet = writer.sheets['Статистика']
        
        # === Создание форматов для условного форматирования (xlsxwriter делает это надёжно) ===
        greenFormat = workbook.add_format({'bg_color': "#D1F5D8"})
        redFormat = workbook.add_format({'bg_color': "#FACCD1"})
        
        # === Применяем условное форматирование ===
        for col_letter in ['C', 'D', 'G', 'I', 'J', 'K', 'L']:
            worksheet.conditional_format(
                f'{col_letter}3:{col_letter}{last_row}',
                {'type': 'formula', 'criteria': f'={col_letter}3>{col_letter}2', 'format': greenFormat}
            )
            worksheet.conditional_format(
                f'{col_letter}3:{col_letter}{last_row}',
                {'type': 'formula', 'criteria': f'={col_letter}3<{col_letter}2', 'format': redFormat}
            )
        logger.info(f'Применено условное фомратирование')
        
        # === Автоширина столбцов ===
        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(str(col))) + 2
            worksheet.set_column(i, i, min(max_len, 32))
        logger.info(f'Изменена ширина столбцов')
        
        # === ВАЖНО: Закрываем xlsxwriter ПОЛНОСТЬЮ перед openpyxl ===
        writer.close()
        
        # === Применяем числовые форматы с помощью openpyxl ===
        wb = load_workbook(file_path)
        ws = wb.active
        
        # === Подставим формулу рассчета CTR ===
        for row in range(2, last_row + 1):
            ws[f'J{row}'] = f'=IFERROR(H{row}/G{row},0)'
        logger.info(f'Подставлены формулы')
        
        # === Форматы чисел ===
        rub_format = '#,##0 "₽"'
        count_format = '#,##0'
        percent_format = '0.00%'
        
        # === Форматы по столбцам ===
        column_formats = {
            'A': None,         # Обновлено
            'B': None,         # Дата
            'C': count_format, # Заказано, шт.
            'D': rub_format,   # Заказано, руб.
            'E': count_format, # Выкуплено, шт.
            'F': rub_format,   # Выкуплено, руб.
            'G': count_format, # Показов
            'H': count_format, # Переходов
            'I': count_format, # Добавлений в корзину
            'J': percent_format, # Конверсия в клик
            'K': percent_format, # Конверсия в корзину
            'L': percent_format, # Конверсия в заказ
            'M': percent_format, # Процент выкупа
            'N': count_format, # В отложенные
        }
        
        # === Применяем форматы к ячейкам данных (строки со 2 до last_row) ===
        for col_letter, fmt in column_formats.items():
            if fmt:
                for row in range(2, last_row + 1):
                    ws[f'{col_letter}{row}'].number_format = fmt
        logger.info(f'Форматы чисел изменены')
        
        # === Границы для всех ячеек ===
        thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )
        
        for row in ws.iter_rows(min_row=1, max_row=last_row, min_col=1, max_col=ws.max_column):
            for cell in row:
                cell.border = thin_border
        logger.info(f'Границы преобразованы')
        
        # === Форматы заголовка ===
        header_font = Font(bold=True, color='000000')
        header_fill = PatternFill(start_color='CCECFF', fill_type='solid')
        center_align = Alignment(horizontal='center', vertical='center')
        
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = center_align
        logger.info(f'Заголовки преобразованы')
        
        # === Сохраняем и закрываем файл ===
        wb.save(file_path)
        wb.close()
        logger.success(f'Файл {file} успешно форматирован и сохранен!')
        return True
    except Exception as e:
        logger.error(f'Произошла ошибка на одном из этапов форматирования файла {file}! | {e}')              
                
### ------------------------------------------------- ###      
      
def UPDATE_DAILY_DETAIL_HISTORY_REPORT_CSV(dates: list[str], TOKEN_NAME: str, file_name: str):
    """
    Функция для ежедневного обновления отчета Воронки продаж
    """
    # === Инициализируем токен ===
    try:
        load_dotenv()
        TOKEN_KEY = os.getenv(TOKEN_NAME)
        if TOKEN_KEY == None:
            logger.error('Ошибка инициализации токена!')
            return None
        logger.info('Токен инициализирован!')
        
        for date in dates:
            new_daily_stats = GET_DAILY_DETAIL_HISTORY_REPORT_CSV(date, TOKEN_KEY)
            new_daily_stats = CALCULATE_DAILY_DETAIL_HISTORY_REPORT_CSV(df=new_daily_stats)
                    
            # === Получение сохраненных данных ===
            df_existing = pd.read_excel(file_name)
            
            # === Переименовываем данные для удобства ===
            header_cols = {
            'Обновлено': 'updateDate',
            'Дата': 'reportDate',
            'Заказано, шт.': 'ordersCount',
            'Заказано, руб.': 'ordersSum',
            'Выкуплено, шт.': 'buyoutsCount',
            'Выкуплено, руб.': 'buyoutsSum',
            'Показов': 'showsCount',
            'Переходов в карточки': 'openCard',
            'Добавлений в корзину': 'addToCart',
            'Средняя конверсия в клик': 'showToClickConversion',
            'Средняя конверсия в корзину': 'addToCartConversion',
            'Средняя конверсия в заказ': 'cartToOrderConversion',
            'Средний процент выкупа': 'buyoutPercent',
            'Добавлений в Отложенные': 'addToWishlist'
            }
            df_existing = df_existing.rename(columns=header_cols)
            
            # === Вставка новых данных ===
            if new_daily_stats['reportDate'] not in df_existing['reportDate'].values:
                df_new_day_stats = pd.DataFrame([new_daily_stats], index=None)
                df_combined = pd.concat([df_existing, df_new_day_stats], ignore_index=True)
                df_combined.to_excel(file_name, index=False, sheet_name='Статистика')
                logger.success(f"Данные за {date} добавлены в файл {file_name}")
            
            # === Замена устаревших данных ===
            elif new_daily_stats['reportDate'] in df_existing['reportDate'].values:
                # === Выбираем фрейм с датой обрабатываемой на данный момент ===
                df_updatableDate = df_existing[df_existing['reportDate'] == date]
                # === Заменяем данные, если хотя бы какой-то из показателей увеличился ===
                if (new_daily_stats['ordersCount'] != df_updatableDate['ordersCount'].iloc[0] or 
                    new_daily_stats['ordersSum'] != df_updatableDate['ordersSum'].iloc[0] or
                    new_daily_stats['buyoutsCount'] != df_updatableDate['buyoutsCount'].iloc[0] or
                    new_daily_stats['buyoutsSum'] != df_updatableDate['buyoutsSum'].iloc[0] or
                    new_daily_stats['openCard'] != df_updatableDate['openCard'].iloc[0] or
                    new_daily_stats['addToCart'] != df_updatableDate['addToCart'].iloc[0] or
                    new_daily_stats['addToWishlist'] != df_updatableDate['addToWishlist'].iloc[0]):
                        
                    # === Т.к Показы увеличиться не могут и по API не тянутся, обновлять их не будем ===
                    showsCount = df_updatableDate['showsCount'].iloc[0]
                    showToClickConversion = df_updatableDate['showToClickConversion'].iloc[0]
                    if showsCount > 0 or showToClickConversion > 0:
                        new_daily_stats['showsCount'] = showsCount 
                        new_daily_stats['showToClickConversion'] = showToClickConversion
                        
                    df_new_day_stats = pd.DataFrame([new_daily_stats], index=None)
                    df_existing = df_existing[df_existing['reportDate'] != date]
                    df_combined = pd.concat([df_existing, df_new_day_stats], ignore_index=True)
                    df_combined.to_excel(file_name, index=False, sheet_name='Статистика')
                         
                    logger.success(f"Данные за {date} в файле {file_name} успешно обновлены")
                else:
                    logger.success(f"Данные за {date} в файле {file_name} уже актуальны")
            else:
                logger.critical(f'Неопознанная ошибка!!!')
                
            if date != dates[-1]:
                logger.info(f'Ожидание 30 секунд для предотвращения ошибки 429...')
                time.sleep(30)
        
        if(FORMAT_DAILY_DETAIL_HISTORY_REPORT_CSV(file=file_name)):
            logger.success(f'Форматированние данных при обновлении периода {dates[0]} - {dates[-1]} прошло успешно')
        logger.success(f'Данные за период {dates[0]} - {dates[-1]} успешно обновлены')
    
    except Exception as e:
        logger.error(f'Возникла ошибка на одном из этапов обновления данных | {e}')    

### ------------------------------------------------- ### 
           
def GET_DAILY_DETAIL_HISTORY_REPORT(TOKEN_NAME: str, date: str, max_attempts: int = 5) -> list:
    """
    Функция получения данных из Воронки продаж
    """
    logger.info(f'Получение статистики из Воронки продаж за {date}')
    load_dotenv()
    TOKEN_KEY = os.getenv(TOKEN_NAME)
    HEADERS = {'Authorization': TOKEN_KEY}
    url = 'https://seller-analytics-api.wildberries.ru/api/analytics/v3/sales-funnel/products'
    dateStartEnd =  str(datetime.strptime(date, "%d.%m.%Y").strftime("%Y-%m-%d"))
    params = {
        'selectedPeriod': {
            'start': dateStartEnd,
            'end': dateStartEnd
        },
        'skipDeletedNm': False,
        'orderBy': {
            'field': 'openCard',
            'mode': 'desc'
        },
        'limit': 1000
    }
    for attempt in range(max_attempts):
        response = requests.post(url=url, headers=HEADERS, json=params)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 429:
            if attempt == max_attempts - 1:
                logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                return None
            logger.info(f'{STATUS_CODE} | {response.text} | Ожидание {5*(attempt+1)}сек.')
            time.sleep(5 * (attempt+1))
        elif response.status_code in [400, 401, 402, 403]:
            logger.error(f'{response.status_code} | {response.text}')
            return None
        else:
            logger.debug(f'Данные из Воронки продаж за {date} получены')
            break   
    products = response.json().get('data').get('products')

    detail_history = []
    for product in products:
        detail_history_product = {}
        detail_history_product['nmId'] = product.get('product').get('nmId')
        detail_history_product['title'] = product.get('product').get('title')
        detail_history_product['vendorCode'] = product.get('product').get('vendorCode')
        detail_history_product['openCount'] = product.get('statistic').get('selected').get('openCount')
        detail_history_product['cartCount'] = product.get('statistic').get('selected').get('cartCount')
        detail_history_product['orderCount - cancelCount'] = product.get('statistic').get('selected').get('orderCount') - product.get('statistic').get('selected').get('cancelCount')
        detail_history_product['orderSum - cancelSum'] = product.get('statistic').get('selected').get('orderSum') - product.get('statistic').get('selected').get('cancelSum')
        detail_history_product['buyoutCount'] = product.get('statistic').get('selected').get('buyoutCount')
        detail_history_product['buyoutSum'] = product.get('statistic').get('selected').get('buyoutSum')
        detail_history.append(detail_history_product)
        
    if not detail_history:
        logger.error(f'Возникла ошибка при обработке данных за {date}')
        return None
    logger.success(f'Данные из Воронки продаж за {date} обработаны!')
    return detail_history
       
### ------------------------------------------------- ###

def GET_REALIZATION_DETAIL_REPORT(TOKEN_NAME: str, date: str, max_attempts: int = 5) -> list:    
    """
    Функция для получения отчета о реализации \n
    date в формате dd.mm.yyyy
    """
    logger.debug(f'Получение отчета о реализации за {date}')
    
    load_dotenv()
    TOKEN_KEY = os.getenv(TOKEN_NAME)
    HEADERS = {'Authorization': TOKEN_KEY}
    
    url = 'https://statistics-api.wildberries.ru/api/v5/supplier/reportDetailByPeriod'
    
    dateFromTo = str(datetime.strptime(date, "%d.%m.%Y").strftime("%Y-%m-%d"))
    params = {
        'dateFrom': dateFromTo,
        'dateTo': dateFromTo,
        'period': 'daily'
    }
    
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS, params=params)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 429:
            WAIT_TIME = 5 * (attempt + 1)
            if attempt == (max_attempts - 1):
                logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                return None
            logger.warning(f'Ошибка {STATUS_CODE}, ожидание {WAIT_TIME} секунд.')
            time.sleep(WAIT_TIME)
        elif STATUS_CODE == 204:
            logger.warning('Нет данных!')
            return None
        elif STATUS_CODE != 200:
            logger.error(f'Ошибка {STATUS_CODE} | {response.text}')
            return None
        else:
            data = response.json()
            logger.success('Все данные получены!')
            break
    
    if not data:
        logger.warning(f'Отчет за {date} пуст !!!')
        return None
    data = response.json()
    logger.success(f'Отчет за {date} получен!')
    
    realization_detail_report = []
    for el in data:
        realization_detail_el = {}
        realization_detail_el['subject_name'] = el.get('subject_name')
        realization_detail_el['nm_id'] = el.get('nm_id')
        realization_detail_el['barcode'] = el.get('barcode')
        realization_detail_el['doc_type_name'] = el.get('doc_type_name')
        realization_detail_el['quantity'] = el.get('quantity')
        realization_detail_el['retail_price'] = el.get('retail_price')
        realization_detail_el['retail_amount'] = el.get('retail_amount')
        realization_detail_el['sale_percent'] = el.get('sale_percent')
        realization_detail_el['commission_percent'] = el.get('commission_percent')
        realization_detail_el['supplier_oper_name'] = el.get('supplier_oper_name')
        realization_detail_el['retail_price_withdisc_rub'] = el.get('retail_price_withdisc_rub')
        realization_detail_el['delivery_amount'] = el.get('delivery_amount')
        realization_detail_el['return_amount'] = el.get('return_amount')
        realization_detail_el['delivery_rub'] = el.get('delivery_rub')
        realization_detail_el['product_discount_for_report'] = el.get('product_discount_for_report')
        realization_detail_el['supplier_promo'] = el.get('supplier_promo')
        realization_detail_el['ppvz_spp_prc'] = el.get('ppvz_spp_prc')
        realization_detail_el['ppvz_kvw_prc_base'] = el.get('ppvz_kvw_prc_base')
        realization_detail_el['ppvz_kvw_prc'] = el.get('ppvz_kvw_prc')
        realization_detail_el['sup_rating_prc_up'] = el.get('sup_rating_prc_up')
        realization_detail_el['is_kgvp_v2'] = el.get('is_kgvp_v2')
        realization_detail_el['ppvz_sales_commission'] = el.get('ppvz_sales_commission')
        realization_detail_el['ppvz_for_pay'] = el.get('ppvz_for_pay')
        realization_detail_el['ppvz_reward'] = el.get('ppvz_reward')
        realization_detail_el['acquiring_fee'] = el.get('acquiring_fee')
        realization_detail_el['acquiring_percent'] = el.get('acquiring_percent')
        realization_detail_el['payment_processing'] = el.get('payment_processing')
        realization_detail_el['ppvz_vw'] = el.get('ppvz_vw')
        realization_detail_el['bonus_type_name'] = el.get('bonus_type_name', None)
        realization_detail_el['penalty'] = el.get('penalty')
        realization_detail_el['additional_payment'] = el.get('additional_payment')
        realization_detail_el['rebill_logistic_cost'] = el.get('rebill_logistic_cost')
        realization_detail_el['storage_fee'] = el.get('storage_fee')
        realization_detail_el['deduction'] = el.get('deduction')
        realization_detail_el['acceptance'] = el.get('acceptance')
        realization_detail_el['srid'] = el.get('srid')
        realization_detail_el['installment_cofinancing_amount'] = el.get('installment_cofinancing_amount')
        realization_detail_el['cashback_amount'] = el.get('cashback_amount')
        realization_detail_el['cashback_discount'] = el.get('cashback_discount')
        realization_detail_el['cashback_commission_change'] = el.get('cashback_commission_change')
        realization_detail_el['order_uid'] = el.get('order_uid')
        realization_detail_el['payment_schedule'] = el.get('payment_schedule')
        realization_detail_el['seller_promo_discount'] = el.get('seller_promo_discount')
        realization_detail_el['loyalty_discount'] = el.get('loyalty_discount')
        realization_detail_el['sale_price_promocode_discount_prc'] = el.get('sale_price_promocode_discount_prc')
        realization_detail_el['sale_price_affiliated_discount_prc'] = el.get('sale_price_affiliated_discount_prc')
        
        realization_detail_report.append(realization_detail_el)
        
    return realization_detail_report
    
### ------------------------------------------------- ### 

def GET_PAID_STORAGE(TOKEN_NAME: str, date: str, max_attempts: int = 5, DEF_WAIT: int = 5) -> list:
    """
    Функция для получения отчета о платном хранении \n
    date в формате dd.mm.yyyy
    """
    logger.debug(f'Получение отчета о платном хранении за {date}')
    
    load_dotenv()
    TOKEN_KEY = os.getenv(TOKEN_NAME)
    HEADERS = {'Authorization': TOKEN_KEY}
    
    url = 'https://seller-analytics-api.wildberries.ru/api/v1/paid_storage'
    
    dateFromTo = str(datetime.strptime(date, "%d.%m.%Y").strftime("%Y-%m-%d"))
    params = {
        'dateFrom': dateFromTo,
        'dateTo': dateFromTo,
    }
    
    logger.info(f'Создание задания на формирование отчета о платном хранении за {date}')
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS, params=params)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 429:
            WAIT_TIME = 5 * (attempt + 1)
            if attempt == (max_attempts - 1):
                logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                return None
            logger.warning(f'Ошибка {STATUS_CODE}, ожидание {WAIT_TIME} секунд.')
            time.sleep(WAIT_TIME)
        elif STATUS_CODE != 200:
            logger.error(f'Ошибка {STATUS_CODE} | {response.text}')
            return None
        else:
            taskId = response.json().get('data').get('taskId')
            logger.debug(f'Задание на генерацию отчета о платном хранении за {date} создано!')
            break
        
    url = f'https://seller-analytics-api.wildberries.ru/api/v1/paid_storage/tasks/{taskId}/status'

    time.sleep(DEF_WAIT)
    logger.info(f'Проверка статуса отчета о платном хранении за {date}')
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 200:
            status = response.json().get('data').get('status')
            if status == 'done':
                break
            elif status == 'new' or status == 'processing':
                WAIT_TIME = DEF_WAIT * (attempt + 1)
                logger.warning(f'Отчет формируется, ожидание {WAIT_TIME} сек.')
                time.sleep(WAIT_TIME)
                continue
            else:
                logger.error(f'Статус отчета {status}, попробуйте повторить позже')
                return None
        elif STATUS_CODE == 429:
            WAIT_TIME = DEF_WAIT * (attempt + 1)
            if attempt == (max_attempts - 1):
                logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                return None
            logger.warning(f'Ошибка {STATUS_CODE}, ожидание {WAIT_TIME} секунд.')
            time.sleep(WAIT_TIME)
        else:
            logger.error(f'Ошибка {STATUS_CODE} | {response.text}')
            return None
        
    url = f'https://seller-analytics-api.wildberries.ru/api/v1/paid_storage/tasks/{taskId}/download'

    logger.info(f'Получение отчета о платном хранении за {date}')
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 200:
            data = response.json()
            logger.debug(f'Отчет о платном хранении за {date} получен!')
            break
        elif STATUS_CODE == 429:
            WAIT_TIME = DEF_WAIT * (attempt + 1)
            if attempt == (max_attempts - 1):
                logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                return None
            logger.warning(f'Ошибка {STATUS_CODE}, ожидание {WAIT_TIME} секунд.')
            time.sleep(WAIT_TIME)
        else:
            logger.error(f'Ошибка {STATUS_CODE} | {response.text}')
            return None
        
    if data:
        logger.info(f'Обработка отчета о платном хранении за {date}')
        paid_storage_df = pd.DataFrame(data, index=None)
        paid_storage_df = (paid_storage_df.groupby(['vendorCode', 'nmId'])['warehousePrice'].sum()).reset_index()
        paid_storage = paid_storage_df.to_dict(orient='records')
        logger.success(f'Отчет о платном хранении за {date} готов!')
        return paid_storage
    else:
        logger.error(f'Ошибка получения отчета о платном хранении за {date}!')
        return None
    
### ------------------------------------------------- ### 

def GET_COST_PRICE(file_path: str = 'required_files/Себестоимости.xlsx') -> list:
    """
    Функция получения себестоимостей из файла Excel
    """
    logger.info(f'Получение себестоимостей')
    try:
        cost_price_df = pd.read_excel(file_path, index_col= None)
        cost_price_df = cost_price_df.rename(columns={
            'Код': 'vendorCode',
            'Наименование': 'title',
            'Себестоимость': 'costPrice',
        })
        cost_price = cost_price_df.to_dict(orient='records')
        logger.success('Себестоимости успешно получены!')
        return cost_price
    except:
        logger.error('Возникла проблема получения себестоимостей товаров!')
        return None

### ------------------------------------------------- ### 

def GET_ADVERTS_LIST(TOKEN_NAME: str, max_attempts: int = 5, DEF_WAIT = 5) -> list:
    """
    Функция возвращает список созданных в кабинете РК
    """
    logger.debug(f'Получение списка рекламных кампаний')
    
    load_dotenv()
    TOKEN_KEY = os.getenv(TOKEN_NAME)
    HEADERS = {'Authorization': TOKEN_KEY}
    
    url = 'https://advert-api.wildberries.ru/adv/v1/promotion/count'
    
    for attempt in range(max_attempts):
        response = requests.get(url=url, headers=HEADERS)
        STATUS_CODE = response.status_code
        if STATUS_CODE == 429:
            WAIT_TIME = 5 * (attempt + 1)
            if attempt == (max_attempts - 1):
                logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                return None
            logger.warning(f'Ошибка {STATUS_CODE}, ожидание {WAIT_TIME} секунд.')
            time.sleep(WAIT_TIME)
        elif STATUS_CODE != 200:
            logger.error(f'Ошибка {STATUS_CODE} | {response.text}')
            return None
        else:
            advert_list = response.json().get('adverts')
            advertsIDs = [advert.get('advertId', None) for group in advert_list for advert in group.get('advert_list', [])]
            logger.debug(f'Список РК получен!')
            return advertsIDs

### ------------------------------------------------- ###

def GET_ADVERTS_STATISTIC(TOKEN_NAME: str, date: str, adverts: list = [], max_attempts: int = 5, DEF_WAIT = 5) -> list:
    """
    Функция возвращает статистику всех РК в кабинете
    """
    logger.debug(f'Получение статистики рекламных кампаний')
    
    load_dotenv()
    TOKEN_KEY = os.getenv(TOKEN_NAME)
    HEADERS = {'Authorization': TOKEN_KEY}
    dateFromTo = str(datetime.strptime(date, "%d.%m.%Y").strftime("%Y-%m-%d"))
    
    url = 'https://advert-api.wildberries.ru/adv/v3/fullstats'
    
    chunk_size = 50
    advertsIDs_chunks = [adverts[i:i + chunk_size] for i in range(0, len(adverts), chunk_size)]
    adverts_full_stat = []
    for adverts_chunk in advertsIDs_chunks:
        adverts_chunk_str = ','.join(map(str, adverts_chunk)) 
        params = {
            'ids': adverts_chunk_str,
            'beginDate': dateFromTo,
            'endDate': dateFromTo
        }
        for attempt in range(max_attempts):
            response = requests.get(url=url, headers=HEADERS, params=params)
            STATUS_CODE = response.status_code
            if STATUS_CODE == 429:
                WAIT_TIME = 5 * (attempt + 1)
                if attempt == (max_attempts - 1):
                    logger.error(f'{STATUS_CODE} | Превышен лимит запросов. Повторите попытку позже.')
                    return None
                logger.warning(f'Ошибка {STATUS_CODE}, ожидание {WAIT_TIME} секунд.')
                time.sleep(WAIT_TIME)
            elif STATUS_CODE != 200:
                logger.error(f'Ошибка {STATUS_CODE} | {response.text}')
                return None
            else:
                if response.json() is not None:
                    adverts_full_stat += response.json()
                    logger.debug(f'Статистика части РК за {date} добавлена')
                else:
                    logger.debug(f'Статистика части РК за {date} отсутствует')
                break
            
        if adverts_chunk != advertsIDs_chunks[-1]:
            logger.info(f'Ожидание 20 сек. для предотвращения ошибки 429')
            time.sleep(20)
            
    if adverts_full_stat:
        logger.success(f'Полная статистика РК за {date} получена')
        full_stats = []
        for advertId in adverts_full_stat:
            for day in advertId.get('days'):
                for app in day.get('apps'):
                    for nm in app.get('nms'):
                        nm_stats = {}
                        nm_stats['nmId'] = nm.get('nmId')
                        nm_stats['sum'] = nm.get('sum')
                        nm_stats['atbs'] = nm.get('atbs')
                        nm_stats['canceled'] = nm.get('canceled')
                        nm_stats['clicks'] = nm.get('clicks')
                        nm_stats['orders'] = nm.get('orders')
                        nm_stats['views'] = nm.get('views')
                        nm_stats['ordersSum'] = nm.get('sum_price')
                        full_stats.append(nm_stats)
        full_stats_df = pd.DataFrame(full_stats, index=None)
        full_stats_df = full_stats_df.groupby('nmId').agg(
            sum = ('sum', 'sum'),
            atbs = ('atbs', 'sum'),
            canceled = ('canceled', 'sum'),
            clicks = ('clicks', 'sum'),
            orders = ('orders', 'sum'),
            views = ('views', 'sum'),
            sum_price = ('ordersSum', 'sum'),
        ).reset_index()
        full_stats_df['orders'] = full_stats_df['orders'] - full_stats_df['canceled']
        full_stats_df = full_stats_df.drop(columns=['canceled'])
        adverts_stats = full_stats_df.to_dict(orient='records')
        return adverts_stats
    else:
        logger.success(f'Ошибка получения рекламной статистики за {date}')
        return None
        

In [2]:
TOKEN_NAME = 'КОСТРИК'

In [3]:
nomenclature_df = pd.DataFrame(APIRequests(TOKEN_NAME=TOKEN_NAME).get_nomenclature(), index=None)

2026-04-14 16:35:12.390 | INFO     | main:__init__:37 - Токен КОСТРИК инициализирован
2026-04-14 16:35:12.391 | INFO     | main:get_nomenclature:70 - Запрос URL: https://content-api.wildberries.ru/content/v2/get/cards/list, попытка: 1/5
2026-04-14 16:35:13.558 | INFO     | main:get_nomenclature:98 - Получено карточек: 100
2026-04-14 16:35:13.559 | INFO     | main:get_nomenclature:70 - Запрос URL: https://content-api.wildberries.ru/content/v2/get/cards/list, попытка: 1/5
2026-04-14 16:35:14.984 | INFO     | main:get_nomenclature:98 - Получено карточек: 117


In [4]:
nomenclature_df

,brand,subjectName,nmID,vendorCode,title,skus
0,PARFUMS CONSTANTINE,Туалетная вода,420728273,GALVANOL000007,Туалетная вода мужская Galvanoliori ULTIMATE G...,[4607817712148]
1,PARFUMS CONSTANTINE,Туалетная вода,420736595,GALVANOL000008,Туалетная вода мужская Galvanoliori LIGHT MOSS...,[4607817712155]
2,PARFUMS CONSTANTINE,Духи,466504507,PCMINI000002,"Духи мужские Gentleman Classic, набор пробнико...",[4607817711660]
3,PARFUMS CONSTANTINE,Духи,245202461,BOHEMIAS000001,Духи женские стойкие BOHEMIA набор пробников 5...,[4607817711240]
4,PARFUMS CONSTANTINE,Духи,612997470,BOHEMIAB000006,"Духи женские набор BOHEMIA BLACK & WHITE, 4 шт...",[4607817712582]
...,...,...,...,...,...,...
112,PARFUMS CONSTANTINE,Гели,543758732,PCUHOD000005,Парфюмированный гель для душа BOHEMIA IN BLACK...,[4601364081658]
113,PARFUMS CONSTANTINE,Гели,543822698,PCUHOD000004,Парфюмированный гель для душа BOHEMIA MAGNIFIC...,[4601364081641]
114,PARFUMS CONSTANTINE,Гели,543810697,PCUHOD000003,Парфюмированный гель для душа BOHEMIA ILLUSION...,[4601364081634]
115,PARFUMS CONSTANTINE,Гели,545066244,PCUHOD000006,Парфюмированный гель для душа BOHEMIA MIDNIGHT...,[4601364081665]


KeyError: ('nm_id', 'ppvz_spp_prc')

,nm_id,retail_price,delivery_rub,commission_percent,spp
7,12190436,25008.60,735.67,14.023810,16.245000
12,12190441,20995.95,840.32,14.794118,17.016471
2,12190430,14969.04,475.41,13.478261,15.514783
9,12190438,9284.00,304.65,15.730000,18.401429
21,21013821,8975.95,491.19,12.400000,13.384667
...,...,...,...,...,...
59,543804384,0.00,117.80,0.000000,0.000000
49,411365420,0.00,122.00,0.000000,0.000000
42,101236421,0.00,0.00,0.000000,0.000000
4,12190433,0.00,0.00,0.000000,0.000000


In [19]:
REALIZATION_DETAIL_df

,subject_name,nm_id,barcode,doc_type_name,quantity,retail_price,retail_amount,sale_percent,commission_percent,supplier_oper_name,...,installment_cofinancing_amount,cashback_amount,cashback_discount,cashback_commission_change,order_uid,payment_schedule,seller_promo_discount,loyalty_discount,sale_price_promocode_discount_prc,sale_price_affiliated_discount_prc
0,Туалетная вода,21013824,4603720462231,,0,0.00,0.00,0,0.0,Логистика,...,0,0,0.0,0.0,r424f7c8cb30e49458090bada8490ff63,0,0,0,0,0
1,Туалетная вода,21013824,4603720462231,Продажа,1,1290.00,861.00,0,31.0,Продажа,...,0,0,0.0,0.0,r424f7c8cb30e49458090bada8490ff63,0,0,0,0,0
2,Туалетная вода,12190436,4603720462095,,0,0.00,0.00,0,0.0,Логистика,...,0,0,0.0,0.0,256256358_7580995240541629646,0,0,0,0,0
3,Туалетная вода,12190436,4603720462095,Продажа,1,1320.00,881.00,0,31.0,Продажа,...,0,0,0.0,0.0,256256358_7580995240541629646,0,0,0,0,0
4,Туалетная вода,12190436,4603720462095,,0,0.00,0.00,0,0.0,Логистика,...,0,0,0.0,0.0,rac44c4b30c6649dcb66645b0b74bfe0c,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,Туалетная вода,23791015,4603720462385,Продажа,1,925.04,688.22,0,31.0,Продажа,...,0,0,0.0,0.0,radf6528f22244eee9da7210d597a9e92,0,0,0,0,0
654,Туалетная вода,21013822,4603720462217,,0,0.00,0.00,0,0.0,Логистика,...,0,0,0.0,0.0,rdfc786170617464fb75a80296e4c3dcb,0,0,0,0,0
655,Туалетная вода,21013822,4603720462217,Продажа,1,1305.70,941.79,0,31.0,Продажа,...,0,0,0.0,0.0,rdfc786170617464fb75a80296e4c3dcb,0,0,0,0,0
656,Туалетная вода,21013821,4603720462149,,0,0.00,0.00,0,0.0,Логистика,...,0,0,0.0,0.0,iaaa9296b0551af118c82a4d0f8bcde06,0,0,0,0,0


In [114]:
DAILY_DETAIL_HISTORY_df = pd.DataFrame(GET_DAILY_DETAIL_HISTORY_REPORT(TOKEN_NAME=TOKEN_NAME, date='04.04.2026'), index=None)

2026-04-14 15:54:09.215 | INFO     | __main__:GET_DAILY_DETAIL_HISTORY_REPORT:439 - Получение статистики из Воронки продаж за 04.04.2026
2026-04-14 15:54:10.648 | DEBUG    | __main__:GET_DAILY_DETAIL_HISTORY_REPORT:470 - Данные из Воронки продаж за 04.04.2026 получены
2026-04-14 15:54:10.664 | SUCCESS  | __main__:GET_DAILY_DETAIL_HISTORY_REPORT:491 - Данные из Воронки продаж за 04.04.2026 обработаны!


In [115]:
DAILY_DETAIL_HISTORY_df

,nmId,title,vendorCode,openCount,cartCount,orderCount - cancelCount,orderSum - cancelSum,buyoutCount,buyoutSum
0,12190436,Духи женские императрица со шлейфом Mademoisel...,INTERSHL000009,658,98,20,26400,20,26400
1,12190441,Духи женские стойкие со шлейфом Mademoiselle 7...,PARFCONS000002,517,58,17,22185,15,19575
2,415555134,Духи зеленые цветочные Eclat Galvanoliori LILA...,GALVANOL000003,327,63,10,8200,9,7380
3,23791011,"Духи женские со шлейфом сирень New York SEVEN,...",ELOR62347,319,31,7,7525,7,7525
4,23791014,"Духи женские Good Girl ванильные New York TEN,...",ELOR62378,309,39,11,11825,10,10750
...,...,...,...,...,...,...,...,...,...
115,545087218,Парфюмированный гель для душа BOHEMIA NIGHT DR...,PCUHOD000009,1,1,0,0,0,0
116,12645298,"Духи ароматы Milano, 60мл парфюмерная вода",PCLM000005,0,0,0,0,0,0
117,547837924,Лимитированный подарочный набор для женщин BOH...,BOHEMIAS000007,0,0,0,0,0,0
118,547857051,Лимитированный подарочный набор для женщин BOH...,BOHEMIAS000009,0,0,0,0,0,0


In [26]:
REALIZATION_DETAIL_df = pd.DataFrame(GET_REALIZATION_DETAIL_REPORT(TOKEN_NAME=TOKEN_NAME, date='04.04.2026'), index=None)

2026-04-14 17:14:26.604 | DEBUG    | __main__:GET_REALIZATION_DETAIL_REPORT:501 - Получение отчета о реализации за 04.04.2026
2026-04-14 17:14:27.586 | SUCCESS  | __main__:GET_REALIZATION_DETAIL_REPORT:534 - Все данные получены!
2026-04-14 17:14:27.605 | SUCCESS  | __main__:GET_REALIZATION_DETAIL_REPORT:541 - Отчет за 04.04.2026 получен!


In [49]:
spp_df = REALIZATION_DETAIL_df[['nm_id', 'ppvz_spp_prc']][REALIZATION_DETAIL_df[['nm_id', 'ppvz_spp_prc']]['ppvz_spp_prc'] > 0]
spp_df = spp_df.groupby('nm_id')['ppvz_spp_prc'].mean().reset_index()

In [48]:
logistic_df = REALIZATION_DETAIL_df[['nm_id', 'delivery_rub']][REALIZATION_DETAIL_df[['nm_id', 'delivery_rub']]['delivery_rub'] > 0]
logistic_df = logistic_df.groupby('nm_id')['delivery_rub'].sum().reset_index()

In [119]:
PAID_STORAGE_df = pd.DataFrame(GET_PAID_STORAGE(TOKEN_NAME=TOKEN_NAME, date='04.04.2026'), index=None)

2026-04-14 15:55:08.934 | DEBUG    | __main__:GET_PAID_STORAGE:604 - Получение отчета о платном хранении за 04.04.2026
2026-04-14 15:55:08.937 | INFO     | __main__:GET_PAID_STORAGE:618 - Создание задания на формирование отчета о платном хранении за 04.04.2026
2026-04-14 15:55:09.393 | DEBUG    | __main__:GET_PAID_STORAGE:634 - Задание на генерацию отчета о платном хранении за 04.04.2026 создано!
2026-04-14 15:55:14.394 | INFO     | __main__:GET_PAID_STORAGE:640 - Проверка статуса отчета о платном хранении за 04.04.2026
2026-04-14 15:55:14.876 | INFO     | __main__:GET_PAID_STORAGE:669 - Получение отчета о платном хранении за 04.04.2026
2026-04-14 15:55:15.511 | DEBUG    | __main__:GET_PAID_STORAGE:675 - Отчет о платном хранении за 04.04.2026 получен!
2026-04-14 15:55:15.512 | INFO     | __main__:GET_PAID_STORAGE:689 - Обработка отчета о платном хранении за 04.04.2026
2026-04-14 15:55:15.528 | SUCCESS  | __main__:GET_PAID_STORAGE:693 - Отчет о платном хранении за 04.04.2026 готов!


In [120]:
PAID_STORAGE_df

,vendorCode,nmId,warehousePrice
0,BOHEMIA000001,12645288,16.596000
1,BOHEMIA000002,12645289,24.074000
2,BOHEMIA000004,12645291,21.231500
3,BOHEMIA000005,12645292,9.845000
4,BOHEMIA000006,12645293,13.957500
...,...,...,...
109,SELECTIV000013,490485179,1.685376
110,SELECTIV000014,490477519,0.922944
111,SELECTIV000015,490489082,0.762432
112,SELECTIV000016,490516010,0.632016


In [121]:
COST_PRICE_df = pd.DataFrame(GET_COST_PRICE(), index=None)

2026-04-14 15:55:23.590 | INFO     | __main__:GET_COST_PRICE:705 - Получение себестоимостей
2026-04-14 15:55:23.636 | SUCCESS  | __main__:GET_COST_PRICE:714 - Себестоимости успешно получены!


In [122]:
COST_PRICE_df

,vendorCode,title,costPrice
0,GALVANOL000001,Туалетная вода для женщин SIGNATURE CHERRY JEM...,138.0
1,GALVANOL000024,Туалетная вода для женщин SIGNATURE FLORAL WHI...,144.0
2,GALVANOL000005,Туалетная вода для женщин SIGNATURE GOLD TALIS...,141.0
3,GALVANOL000003,Туалетная вода для женщин SIGNATURE LILAC FLEU...,145.0
4,GALVANOL000004,Туалетная вода для женщин SIGNATURE MAGNOLIA D...,143.0
...,...,...,...
340,SELECTIV000014,Парфюмерная вода для женщин SELECTIVE Mystic R...,575.0
341,SELECTIV000013,Парфюмерная вода для женщин SELECTIVE Pure Jas...,601.0
342,SELECTIV000015,Парфюмерная вода для женщин SELECTIVE Romantic...,575.0
343,SELECTIV000017,Парфюмерная вода для женщин SELECTIVE White cr...,580.0


In [124]:
ADVERTS_STATISTIC_df = pd.DataFrame(GET_ADVERTS_STATISTIC(TOKEN_NAME=TOKEN_NAME, date='04.04.2026', adverts=GET_ADVERTS_LIST(TOKEN_NAME=TOKEN_NAME)), index=None)

2026-04-14 15:55:53.772 | DEBUG    | __main__:GET_ADVERTS_LIST:726 - Получение списка рекламных кампаний
2026-04-14 15:55:54.254 | DEBUG    | __main__:GET_ADVERTS_LIST:750 - Список РК получен!
2026-04-14 15:55:54.258 | DEBUG    | __main__:GET_ADVERTS_STATISTIC:759 - Получение статистики рекламных кампаний
2026-04-14 15:55:54.769 | DEBUG    | __main__:GET_ADVERTS_STATISTIC:796 - Статистика части РК за 04.04.2026 отсутствует
2026-04-14 15:55:54.772 | INFO     | __main__:GET_ADVERTS_STATISTIC:800 - Ожидание 20 сек. для предотвращения ошибки 429
2026-04-14 15:56:15.292 | DEBUG    | __main__:GET_ADVERTS_STATISTIC:796 - Статистика части РК за 04.04.2026 отсутствует
2026-04-14 15:56:15.294 | INFO     | __main__:GET_ADVERTS_STATISTIC:800 - Ожидание 20 сек. для предотвращения ошибки 429
2026-04-14 15:56:35.763 | DEBUG    | __main__:GET_ADVERTS_STATISTIC:796 - Статистика части РК за 04.04.2026 отсутствует
2026-04-14 15:56:35.765 | INFO     | __main__:GET_ADVERTS_STATISTIC:800 - Ожидание 20 сек. 

In [125]:
ADVERTS_STATISTIC_df

,nmId,sum,atbs,clicks,orders,views,sum_price
0,12190429,1202.22,17,103,6,3679,8976
1,12190430,148.09,6,12,2,1260,2992
2,12190431,430.59,8,49,3,3207,4488
3,12190432,0.24,2,0,0,3,0
4,12190433,251.80,6,37,2,1782,2992
...,...,...,...,...,...,...,...
57,466545270,0.00,3,0,0,0,0
58,547857051,0.06,0,0,0,1,0
59,577225812,57.94,3,17,0,740,0
60,577260169,39.33,2,4,1,516,621


In [ ]:
full_report = pd.merge(left=nomenclature_df, right=DAILY_DETAIL_HISTORY_df.drop(columns=[]), )